# Supervised Fine-Tuning with PubMed Dataset on Microsoft Foundry

This notebook demonstrates how to fine-tune language models using **Supervised Fine-Tuning (SFT)** with the PubMed Medical Research Summarization dataset.

## What You'll Learn
1. Understand supervised fine-tuning for medical text summarization
2. Prepare and format medical research data
3. Upload datasets to Microsoft Foundry
4. Create and monitor a supervised fine-tuning job
5. Deploy and test your fine-tuned model

**Note**: Execute each cell in sequence.

## 1. Setup and Installation

Install all required packages from requirements.txt

In [1]:
pip install -r requirements.txt

Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

print("All libraries imported successfully")

All libraries imported successfully


## 3. Configure Azure Environment

Set your Microsoft Foundry Project endpoint and model name. We're using **gpt-4.1** in this example, but you can use other supported GPT models. Copy the file `.env.template` (located in this folder), and save it as file named `.env`. Enter appropriate values for the environment variables used for the job you want to run.

```
MICROSOFT_FOUNDRY_PROJECT_ENDPOINT=<your-endpoint>
MODEL_NAME=gpt-4.1
AZURE_SUBSCRIPTION_ID=<your-subscription-id>
AZURE_RESOURCE_GROUP=<your-resource-group>
AZURE_AOAI_ACCOUNT=<your-foundry-account-name>
```

In [3]:
load_dotenv()

endpoint = os.environ.get("MICROSOFT_FOUNDRY_PROJECT_ENDPOINT")
model_name = os.environ.get("MODEL_NAME")

# Create a bounded demonstration subset so the end-to-end run completes promptly.
from itertools import islice

def write_subset(source_path, destination_path, count, start=0):
    with open(source_path, "r", encoding="utf-8") as source, open(destination_path, "w", encoding="utf-8") as destination:
        destination.writelines(islice(source, start, start + count))

training_file_path = "training_demo.jsonl"
validation_file_path = "validation_demo.jsonl"
write_subset("training.jsonl", training_file_path, 10)
write_subset("training.jsonl", validation_file_path, 10, start=10)
print("Prepared disjoint 10-example training and validation subsets for this demonstration run")

Prepared disjoint 10-example training and validation subsets for this demonstration run


## 4. Connect to Microsoft Foundry Project

Connect to Microsoft Foundry Project using Azure credential authentication. This initializes the project client and OpenAI client needed for fine-tuning workflows.

**Important**: Ensure you have the **Azure AI User** role assigned to your account for the Microsoft Foundry Project resource.

In [4]:
credential = AzureCliCredential(process_timeout=120)
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to Microsoft Foundry Project")

Connected to Microsoft Foundry Project


## 5. Upload Training Files

Upload the training and validation JSONL files to Microsoft Foundry. Each file is assigned a unique ID that will be referenced when creating the fine-tuning job.

In [5]:
print("Uploading training file...")
with open(training_file_path, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Training file ID: {train_file.id}")

print("\nUploading validation file...")
with open(validation_file_path, "rb") as f:
    validation_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Validation file ID: {validation_file.id}")

Uploading training file...


Training file ID: file-f8053e05d2744c5c8c503afaa9a2a415

Uploading validation file...


Validation file ID: file-8035d2901bf4482cb6ab50b875c7a893


## 6. Wait for File Processing

Microsoft Foundry needs to process the uploaded files before they can be used for fine-tuning.

In [6]:
print("Waiting for files to be processed...")
openai_client.files.wait_for_processing(train_file.id)
openai_client.files.wait_for_processing(validation_file.id)
print("Files ready!")

Waiting for files to be processed...


Files ready!


## 7. Create Supervised Fine-Tuning Job

Create a supervised fine-tuning job with your uploaded datasets. Configure the following hyperparameters to control the training process:

**Hyperparameters:**
1. **n_epochs (3)**: Number of complete passes through the training dataset. More epochs can improve performance but may lead to overfitting. Typical range: 1-10.
2. **batch_size (1)**: Number of training examples processed together in each iteration. Smaller batches provide more frequent updates. Typical range: 1-8.
3. **learning_rate_multiplier (1.0)**: Scales the default learning rate. Values < 1.0 make training more conservative, while values > 1.0 speed up learning but may cause instability. Typical range: 0.1-2.0.

**Note**: Adjust these based on your dataset size and quality.

In [7]:
existing_job_id = os.environ.get("FINE_TUNE_JOB_ID")
if existing_job_id:
    print(f"Reusing completed fine-tuning job: {existing_job_id}")
    fine_tune_job = openai_client.fine_tuning.jobs.retrieve(existing_job_id)
else:
    print("Creating supervised fine-tuning job...")
    fine_tune_job = openai_client.fine_tuning.jobs.create(
        model=model_name,
        training_file=train_file.id,
        validation_file=validation_file.id,
        method={
            "type": "supervised",
            "supervised": {"hyperparameters": {"n_epochs": 1, "batch_size": 1, "learning_rate_multiplier": 1.0}},
        },
        extra_body={"trainingType": "GlobalStandard"},
        suffix="pubmed-summarization"
    )

print(f"Fine-tuning job created!")
print(f"Job ID: {fine_tune_job.id}")
print(f"Status: {fine_tune_job.status}")
print(f"Model: {fine_tune_job.model}")

Reusing completed fine-tuning job: ftjob-72ede626606d40eebb7aaa15060a24ee


Fine-tuning job created!
Job ID: ftjob-72ede626606d40eebb7aaa15060a24ee
Status: succeeded
Model: gpt-4.1-2025-04-14


## 8. Monitor Training Progress

Track the status of your fine-tuning job. You can view the current status, and recent training events. Training duration varies based on dataset size, model, and hyperparameters - typically ranging from minutes to several hours.

In [8]:
import time

terminal_statuses = {"succeeded", "failed", "cancelled"}
while True:
    job_status = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)
    print(f"Status: {job_status.status}")
    if job_status.status in terminal_statuses:
        break
    time.sleep(60)

if job_status.status != "succeeded":
    raise RuntimeError(f"Fine-tuning job {fine_tune_job.id} ended with status {job_status.status}: {job_status.error}")

Status: succeeded


## 9. Retrieve Fine-Tuned Model

After the fine-tuning job succeeded, retrieve the fine-tuned model ID. This ID is required to make inference calls with your customized model.

In [9]:
completed_job = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)

if completed_job.status == "succeeded":
    fine_tuned_model_id = completed_job.fine_tuned_model
    print(f"Fine-tuned Model ID: {fine_tuned_model_id}")
else:
    raise RuntimeError(f"Fine-tuning job ended with status {completed_job.status}: {completed_job.error}")

Fine-tuned Model ID: gpt-4.1-2025-04-14.ft-72ede626606d40eebb7aaa15060a24ee-pubmed-summarization


## 10. Deploy the Fine-Tuned Model

Deploy the fine-tuned model to Azure OpenAI as a deployment endpoint. This step is required before making inference calls. The deployment uses GlobalStandard SKU with 50 capacity.

In [10]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, Sku

subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID")
resource_group = os.environ.get("AZURE_RESOURCE_GROUP")
account_name = os.environ.get("AZURE_AOAI_ACCOUNT")

deployment_name = f"pubmed-{fine_tune_job.id[-8:]}"

with CognitiveServicesManagementClient(credential=credential, subscription_id=subscription_id) as cogsvc_client:
    deployment_model = DeploymentModel(format="OpenAI", name=fine_tuned_model_id, version="1")
    deployment_properties = DeploymentProperties(model=deployment_model)
    deployment_sku = Sku(name="GlobalStandard", capacity=100)
    deployment_config = Deployment(properties=deployment_properties, sku=deployment_sku)
    
    print(f"Deploying fine-tuned model: {fine_tuned_model_id}")
    deployment = cogsvc_client.deployments.begin_create_or_update(
        resource_group_name=resource_group,
        account_name=account_name,
        deployment_name=deployment_name,
        deployment=deployment_config,
    )
    
    print("Waiting for deployment to complete...")
    deployment.result()

print(f"Model deployment completed: {deployment_name}")

Deploying fine-tuned model: gpt-4.1-2025-04-14.ft-72ede626606d40eebb7aaa15060a24ee-pubmed-summarization


Waiting for deployment to complete...
Model deployment completed: pubmed-060a24ee


## 11. Test Fine-Tuned Model

Test your fine-tuned model by generating a summary for a sample medical research article.

In [11]:
from azure.identity import get_bearer_token_provider
from openai import AzureOpenAI

azure_endpoint = f"https://{account_name}.cognitiveservices.azure.com/"

client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    azure_ad_token_provider=get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default"),
    api_version="2024-12-01-preview"
)

test_article = """Background: Alzheimer's disease (AD) is a progressive neurodegenerative disorder characterized by cognitive decline and memory loss. Recent studies have suggested that neuroinflammation plays a crucial role in disease pathogenesis. Objective: This study investigated the role of microglial activation and inflammatory cytokines in early-stage Alzheimer's disease progression. Methods: We conducted a longitudinal study of 150 patients with mild cognitive impairment (MCI) over 36 months. Cerebrospinal fluid (CSF) samples were analyzed for inflammatory markers including IL-6, TNF-α, and IL-1β. Brain imaging using PET scans assessed microglial activation. Results: Patients who progressed to AD showed significantly elevated levels of IL-6 (p<0.001) and TNF-α (p<0.01) at baseline compared to stable MCI patients. Microglial activation was observed in the hippocampus and entorhinal cortex regions. Multivariate analysis revealed that combined inflammatory markers predicted AD conversion with 78% accuracy. Conclusion: Elevated neuroinflammatory markers and microglial activation in MCI patients are associated with increased risk of progression to Alzheimer's disease, suggesting potential therapeutic targets for early intervention."""

response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "You are a medical research summarization assistant."},
        {"role": "user", "content": f"Summarize this medical research article:\n\n{test_article}"}
    ],
    temperature=0.7,
    max_tokens=500
)

print(response.choices[0].message.content)

**Summary:**

This longitudinal study followed 150 patients with mild cognitive impairment (MCI) over 36 months to investigate the role of neuroinflammation in early Alzheimer's disease (AD) progression. Researchers measured cerebrospinal fluid (CSF) levels of inflammatory cytokines (IL-6, TNF-α, IL-1β) and used PET scans to assess microglial activation in the brain. Patients who progressed to AD had significantly higher baseline levels of IL-6 and TNF-α and showed increased microglial activation in the hippocampus and entorhinal cortex compared to those with stable MCI. Combined inflammatory markers predicted conversion to AD with 78% accuracy. The findings suggest that elevated neuroinflammatory markers and microglial activation are linked to increased risk of AD progression, highlighting their potential as therapeutic targets for early intervention.


##  Congratulations!

You've successfully fine-tuned a model for medical research summarization using the PubMed dataset!

### Next Steps:
1. **Test with more examples**: Try different medical research articles to evaluate performance
2. **Adjust hyperparameters**: Experiment with different epoch counts, batch sizes, or learning rates
3. **Deploy to production**: Integrate your fine-tuned model into healthcare applications
4. **Fine-tune further**: Use your own medical publications or clinical notes for specialized summarization
5. **Explore use cases**: Apply the model to systematic reviews, literature analysis, or clinical decision support